# Make sess class from raw data for each session and save as pickle

Run after suite2p curation but before anything else.

Currently uses info from sessions_dict.py to loop through sessions and create the sess class, \
synchronizing neural data with behavioral data.

sess pickle files will be named `<scene>_<session>_<scan>.pickle`  \
and saved in `path_dict['preprocessed_root']/sess/<animal>/<date>`.

Set `overwrite` to `True` if you want to overwrite existing .pickle files. Otherwise, you will get an error that the file already exists.

In [35]:
overwrite = True

In [36]:
import os
import numpy as np

from reward_relative import preprocessing as pp
from reward_relative import utilities as ut

import TwoPUtils



%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Specify your path dictionary here.

Copy and rename `path_dict.py` to a new file and edit it with the paths on your system.

In [37]:
from reward_relative.path_dict_KW import path_dictionary as path_dict
path_dict

{'preprocessed_root': '/Users/kayleewilson/SosaRepo/Data',
 'sbx_root': '/media/marielenasosa/T7/2p_raw_data',
 'gdrive_root': '/mnt/gdrive/2P_Data',
 'VR_Data': '/Users/kayleewilson/SosaRepo/Data/VR_Data',
 'git_repo_root': '/Users/kayleewilson/gitrepos',
 'TwoPUtils': '/Users/kayleewilson/gitrepos/TwoPUtils',
 'home': '/Users/kayleewilson',
 'fig_dir': '/Users/kayleewilson/SosaRepo/Data/fig_scratch'}

## Scroll or click to the desired section for:

[Behavior only](#Behavior-only)


Within each section, define animal and iterate through sessions.

While running the below cells, if you get an error that says `DatabaseError: Execution failed on sql 'SELECT * FROM data': no such table: data`,
check that all of your .sqlite files are named properly and have data in them (i.e. `Scene_1.sqlite` instead of `'Scene_1(1).sqlite'`


# Behavior only

In [38]:
from reward_relative.sessions_dict_behavior_only import sosalab as metadata

In [39]:
metadata

{'pp01': ({'date': '2026_06_03',
   'scene': 'FiveTower_Stay',
   'session': 1,
   'scan': nan,
   'exp_day': 1,
   'GD': nan,
   'pregnant': False,
   'rig': 'omen-vr'},
  {'date': '2026_06_04',
   'scene': 'FiveTower_Stay',
   'session': 1,
   'scan': nan,
   'exp_day': 2,
   'GD': nan,
   'pregnant': False,
   'rig': 'omen-vr'},
  {'date': '2026_06_05',
   'scene': 'FiveTower_Switch_BlackoutDelay',
   'session': 2,
   'scan': nan,
   'exp_day': 3,
   'GD': nan,
   'pregnant': False,
   'rig': 'omen-vr'},
  {'date': '2026_06_06',
   'scene': 'FiveTower_Stay_BlackoutDelay',
   'session': 1,
   'scan': nan,
   'exp_day': 4,
   'GD': nan,
   'pregnant': False,
   'rig': 'omen-vr'},
  {'date': '2026_06_12',
   'scene': 'FiveTower_Switch_BlackoutDelay',
   'session': 1,
   'scan': nan,
   'exp_day': 5,
   'GD': nan,
   'pregnant': False,
   'rig': 'omen-vr'},
  {'date': '2026_06_13',
   'scene': 'FiveTower_Stay_BlackoutDelay',
   'session': 1,
   'scan': nan,
   'exp_day': 6,
   'GD': nan

In [40]:
## Define animal
animal = 'ps04'
days = np.arange(0, len(metadata[animal])) # range of days
# days =days[1:3] # optional select subset of days
days

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

### Main cell to create sess

In [41]:
basedir = os.path.join(path_dict['preprocessed_root'], animal)
sbxdir = os.path.join(path_dict['sbx_root'], animal)
vrdir = path_dict['VR_Data']

binary_from_sbxdir = False # only relevant for downsampling
calcium_exists = False

load_suite2p = False
load_scaninfo = False
VR_only = True

trial_matrix_kwargs = []

for i, day in enumerate(days):

    if type(metadata[animal][day]) is not tuple:
        try:
            date = metadata[animal][day]['date']
            scene = metadata[animal][day]['scene']
            rig = metadata[animal][day]['rig']
            session = metadata[animal][day]['session']
            scan_number = metadata[animal][day]['scan']

            sess = pp.create_sess(basedir, sbxdir, vrdir, animal, date, rig, scene, session, scan_number,
                                load_scaninfo=load_scaninfo,
                                load_VR=True,
                                load_suite2p=load_suite2p,
                                load_behavior=True,
                                VR_only=VR_only,                              
                                )

            sess_dir = os.path.join(
                path_dict['preprocessed_root'], 'sess', animal, date)
            os.makedirs(sess_dir, exist_ok=True)
            print(sess_dir)
        except:
            print(f"----------------- \n\n {animal} : day {day} database could not open \n\n------------------------")

        if np.isnan(scan_number):
            scan_number=0
            
        sess_name = '%s_%03d_%03d.pickle' % (scene,
                                             session,
                                             scan_number
                                             )
        # Write sess to pickle file
        ut.write_sess_pickle(sess, sess_dir, sess_name, overwrite=overwrite)

    else:
        print("Iterating through multiple sessions")
        for i in range(len(metadata[animal][day])):
            date = metadata[animal][day][i]['date']
            scene = metadata[animal][day][i]['scene']
            session = metadata[animal][day][i]['session']
            scan_number = metadata[animal][day][i]['scan']

            sess = pp.create_sess(basedir, sbxdir, vrdir, animal, date, scene, session, scan_number,
                                  load_scaninfo=True,
                                  load_VR=True,
                                  load_suite2p=True,
                                  load_behavior=True)

            sess_dir = os.path.join(
                path_dict['preprocessed_root'], 'sess', animal, date)
            os.makedirs(sess_dir, exist_ok=True)
            print(sess_dir)

            sess_name = '%s_%03d_%03d.pickle' % (scene,
                                                 session,
                                                 scan_number,
                                                 )
            # Write sess to pickle file
            ut.write_sess_pickle(
                sess, sess_dir, sess_name, overwrite=overwrite)

/Users/kayleewilson/SosaRepo/Data/VR_Data/ps04/2026_07_06/omen-vr/FiveTower_Stay_2.sqlite
Fixing teleports
(100714, 16)
/Users/kayleewilson/SosaRepo/Data/sess/ps04/2026_07_06
writing FiveTower_Stay_002_000.pickle
/Users/kayleewilson/SosaRepo/Data/VR_Data/ps04/2026_07_07/omen-vr/FiveTower_Stay_2.sqlite
Fixing teleports
(112739, 16)
/Users/kayleewilson/SosaRepo/Data/sess/ps04/2026_07_07
writing FiveTower_Stay_002_000.pickle
/Users/kayleewilson/SosaRepo/Data/VR_Data/ps04/2026_07_08/omen-vr/FiveTower_Switch_BlackoutDelay_1.sqlite
Fixing teleports
(108490, 16)
/Users/kayleewilson/SosaRepo/Data/sess/ps04/2026_07_08
writing FiveTower_Switch_BlackoutDelay_001_000.pickle
/Users/kayleewilson/SosaRepo/Data/VR_Data/ps04/2026_07_09/omen-vr/FiveTower_Stay_BlackoutDelay_1.sqlite
Fixing teleports
(104488, 16)
/Users/kayleewilson/SosaRepo/Data/sess/ps04/2026_07_09
writing FiveTower_Stay_BlackoutDelay_001_000.pickle
/Users/kayleewilson/SosaRepo/Data/VR_Data/ps04/2026_07_17/omen-vr/FiveTower_Switch_Black

In [42]:
sess = ut.load_sess_pickle(path_dict['preprocessed_root'], 'pp01', exp_day=1)

/Users/kayleewilson/SosaRepo/Data/sess/pp01/2026_06_03/FiveTower_Stay_001_000.pickle
